# Oilfield Brine Workflow (Public-Facing, Cleaned Version)

This notebook documents a full workflow for converting publicly available Annual Surface Application Form 15 PDFs into an analyzable dataset. The code is organized into modular sections so each stage can be described clearly in a protocol manuscript and run independently when needed.

This version is kept Google Colab-friendly, uses placeholder paths for public sharing, and preserves multiple extraction options (Nanonets and Qwen) so you can decide later which branches to keep in the final manuscript.


In [ ]:
# Colab setup: run these installs once per runtime.
!pip install -q pdf2image litellm thefuzz python-dotenv openpyxl
!apt-get install -y poppler-utils
!pip install -q git+https://github.com/huggingface/transformers accelerate
!pip install -q qwen-vl-utils[decord]==0.0.8
!pip install -q -U bitsandbytes opencv-python-headless


## 1) Colab setup and project configuration

This section collects the shared paths, runtime settings, and helper functions used throughout the workflow. Keeping configuration in one place makes the notebook easier to follow and much easier to adapt when you run a new county, year range, or extraction batch.


In [ ]:
from __future__ import annotations

import gc
import json
import logging
import os
import re
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence

import pandas as pd
from dateutil import parser as date_parser
from tqdm import tqdm

try:
    from google.colab import drive, userdata  # type: ignore
    IN_COLAB = True
except ImportError:
    drive = None
    userdata = None
    IN_COLAB = False

logging.getLogger("thefuzz").setLevel(logging.ERROR)

PROJECT_ROOT = Path("/content/drive/MyDrive/<PROJECT_FOLDER>")
WORKFLOW_ROOT = PROJECT_ROOT / "Workflow"

PATHS = {
    "raw_pdfs": WORKFLOW_ROOT / "RawData" / "<COUNTY_OR_BATCH_NAME>",
    "images": WORKFLOW_ROOT / "1_Images",
    "images_enhanced": WORKFLOW_ROOT / "2_ImagesEnhanced",
    "ocr_text": WORKFLOW_ROOT / "3_Extraction",
    "parsed_csv": WORKFLOW_ROOT / "4_DataParsing",
    "merged_csv": WORKFLOW_ROOT / "5_MergedData",
    "cleaning_inputs": WORKFLOW_ROOT / "6_DataCleaning",
    "final_output": WORKFLOW_ROOT / "7_FinalData",
}

RUN_STEPS = {
    "convert_pdfs": False,
    "enhance_images": False,
    "run_nanonets": False,
    "run_qwen": False,
    "parse_text": False,
    "merge_csvs": False,
    "clean_and_match": False,
}

QWEN_CONFIG = {
    "model_name": "Qwen/Qwen2.5-VL-7B-Instruct",
    "quantize_4bit": True,
    "max_new_tokens": 2048,
    "input_folder": PATHS["images_enhanced"] / "<IMAGE_SUBFOLDER>",
    "output_folder": PATHS["ocr_text"] / "<OUTPUT_SUBFOLDER>",
}

LLM_CONFIG = {
    "model": "gpt-4o-mini",
    "api_key_env": "OPENAI_API_KEY",
    "colab_secret_name": "OPENAI_API_KEY",
}

MATCHING_CONFIG = {
    "master_list_path": PATHS["cleaning_inputs"] / "Tiger_roads_compound_key.csv",
    "county_col": "County_Name",
    "township_col": "Township_Name",
    "road_col": "FULLNAME",
    "township_threshold": 85,
    "road_threshold": 80,
}


def mount_google_drive(mount_point: str = "/content/drive", force_remount: bool = True) -> None:
    if not IN_COLAB:
        print("Google Drive mount skipped because this is not a Colab runtime.")
        return
    drive.mount(mount_point, force_remount=force_remount)


def ensure_dir(path: Path | str) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def load_api_key(secret_name: str, env_name: str) -> Optional[str]:
    if IN_COLAB and userdata is not None:
        try:
            value = userdata.get(secret_name)
            if value:
                os.environ[env_name] = value
                return value
        except Exception:
            pass

    value = os.getenv(env_name)
    if value:
        return value

    print(f"No API key found for secret '{secret_name}' or env var '{env_name}'.")
    return None


## 2) PDF to image conversion

This section converts multi-page PDFs into page-level image files. That step makes it easier to run OCR consistently and also lets you inspect individual pages if a later extraction step performs poorly.


In [ ]:
def convert_pdf_to_images(pdf_path: Path | str, output_folder: Path | str, dpi: int = 300, image_format: str = "PNG") -> List[Path]:
    from pdf2image import convert_from_path

    pdf_path = Path(pdf_path)
    output_folder = ensure_dir(output_folder)
    pages = convert_from_path(str(pdf_path), dpi=dpi)

    saved_paths = []
    for page_number, page in enumerate(pages, start=1):
        output_path = output_folder / f"{pdf_path.stem}_page_{page_number:03d}.{image_format.lower()}"
        page.save(output_path, format=image_format)
        saved_paths.append(output_path)
    return saved_paths


def batch_convert_pdfs(pdf_folder: Path | str, image_folder: Path | str, dpi: int = 300, image_format: str = "PNG") -> Dict[str, List[Path]]:
    pdf_folder = Path(pdf_folder)
    image_folder = ensure_dir(image_folder)

    pdf_files = sorted(pdf_folder.glob("*.pdf"))
    conversions = {}
    for pdf_path in tqdm(pdf_files, desc="Converting PDFs"):
        conversions[pdf_path.name] = convert_pdf_to_images(pdf_path, image_folder, dpi=dpi, image_format=image_format)

    print(f"Converted {len(conversions)} PDFs into {sum(len(v) for v in conversions.values())} images.")
    return conversions


## 3) Image enhancement for OCR

This section applies a light preprocessing pipeline to improve readability for OCR and vision models. The image operations are intentionally modest so the enhanced files remain close to the original scans while still improving text contrast.


In [ ]:
def apply_super_enhancement(image, upscale_factor: float = 2.0):
    import cv2

    height, width = image.shape[:2]
    new_size = (int(width * upscale_factor), int(height * upscale_factor))
    upscaled = cv2.resize(image, new_size, interpolation=cv2.INTER_CUBIC)

    gray = cv2.cvtColor(upscaled, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (0, 0), sigmaX=1.5)
    sharpened = cv2.addWeighted(gray, 1.5, blurred, -0.5, 0)
    enhanced = cv2.adaptiveThreshold(sharpened, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 10)
    return enhanced


def batch_enhance_images(input_dir: Path | str, output_dir: Path | str) -> List[Path]:
    import cv2

    input_dir = Path(input_dir)
    output_dir = ensure_dir(output_dir)
    image_paths = sorted(list(input_dir.glob("*.png")) + list(input_dir.glob("*.jpg")) + list(input_dir.glob("*.jpeg")))

    saved_paths = []
    for image_path in tqdm(image_paths, desc="Enhancing images"):
        image = cv2.imread(str(image_path))
        if image is None:
            print(f"Skipping unreadable image: {image_path.name}")
            continue

        enhanced = apply_super_enhancement(image)
        output_path = output_dir / image_path.name
        cv2.imwrite(str(output_path), enhanced)
        saved_paths.append(output_path)

    print(f"Enhanced {len(saved_paths)} images.")
    return saved_paths


## 4) OCR: Qwen 2.5-VL

This section preserves the Qwen-based extraction path and supports both the quantized Colab T4 setup and the fuller L4/A100 setup. Grouping pages by document ensures that multi-page reports are saved back out as one text file per source document.


In [ ]:
def clear_memory() -> None:
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    except Exception:
        pass


def load_qwen_model(model_name: str, quantize_4bit: bool = True):
    import torch
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

    model_kwargs = {
        "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
        "device_map": "auto",
    }
    if quantize_4bit:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )

    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_name, **model_kwargs)
    processor = AutoProcessor.from_pretrained(model_name)
    return model, processor


QWEN_TRANSCRIPTION_PROMPT = (
    "You are a precise data entry specialist. "
    "Transcribe the text in this image word-for-word. "
    "Output standard text as Markdown. "
    "Represent the main data grid using HTML tags, but keep handwritten headers and totals outside the table. "
    "Do not wrap the output in code fences. "
    "Do not invent placeholder values. "
    "Read every field, label, and handwritten value as carefully as possible."
)


def process_single_page_with_qwen(model, processor, image_path: Path | str, prompt: str = QWEN_TRANSCRIPTION_PROMPT, max_new_tokens: int = 2048) -> str:
    import torch
    from qwen_vl_utils import process_vision_info

    image_path = Path(image_path)
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": str(image_path)},
            {"type": "text", "text": prompt},
        ],
    }]

    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text_prompt], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")

    if torch.cuda.is_available():
        inputs = {key: value.to(model.device) for key, value in inputs.items()}

    generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    trimmed_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs["input_ids"], generated_ids)]
    decoded = processor.batch_decode(trimmed_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    return decoded[0].strip()


def group_page_images(image_paths: Sequence[Path]) -> Dict[str, List[Path]]:
    grouped = defaultdict(list)
    for image_path in sorted(image_paths):
        document_name = re.sub(r"_page_\d{3}$", "", image_path.stem)
        grouped[document_name].append(image_path)
    return dict(grouped)


def batch_qwen_extraction(input_dir: Path | str, output_dir: Path | str, model_name: str, quantize_4bit: bool = True, max_new_tokens: int = 2048) -> List[Path]:
    input_dir = Path(input_dir)
    output_dir = ensure_dir(output_dir)
    image_paths = sorted(list(input_dir.glob("*.png")) + list(input_dir.glob("*.jpg")) + list(input_dir.glob("*.jpeg")))
    grouped_docs = group_page_images(image_paths)

    model, processor = load_qwen_model(model_name=model_name, quantize_4bit=quantize_4bit)
    saved_paths = []

    for document_name, pages in tqdm(grouped_docs.items(), desc="Qwen extraction"):
        page_text = []
        for page in pages:
            page_result = process_single_page_with_qwen(model=model, processor=processor, image_path=page, max_new_tokens=max_new_tokens)
            page_text.append(f"\n\n# {page.name}\n\n{page_result}")

        output_path = output_dir / f"{document_name}.txt"
        output_path.write_text("".join(page_text).strip(), encoding="utf-8")
        saved_paths.append(output_path)

    clear_memory()
    return saved_paths


## 6) Text-to-table parsing with LiteLLM

This section converts OCR text into structured records that can be analyzed in tabular form. Because the parsing prompt and output schema are central to reproducibility, they are written explicitly so readers can see how the extracted text becomes a dataset.


In [ ]:
TARGET_FIELDS = [
    "source_file",
    "report_year",
    "record_type",
    "operator_name",
    "source_well_name",
    "source_api",
    "app_date",
    "app_county",
    "app_township",
    "app_road",
    "app_volume_bbl",
    "notes",
]

PARSING_SYSTEM_PROMPT = """
You are extracting structured data from OCR text produced from Ohio Annual Surface Application Form 15 reports.
Return JSON only.

Instructions:
- Return a JSON array of records.
- Use one record per logical row in the report.
- Use the field names exactly as listed in the provided schema.
- If a value is missing, use an empty string rather than inventing content.
- Include both spreading records and source-well records if they appear in the text.
- Preserve handwritten values when they can be read confidently.
""".strip()


def extract_json_block(text: str) -> str:
    text = text.strip()
    if text.startswith("[") or text.startswith("{"):
        return text

    array_match = re.search(r"\[.*\]", text, flags=re.DOTALL)
    if array_match:
        return array_match.group(0)

    object_match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if object_match:
        return object_match.group(0)

    raise ValueError("No JSON content found in model response.")


def parse_model_json(text: str) -> List[Dict[str, Any]]:
    parsed = json.loads(extract_json_block(text))
    if isinstance(parsed, dict):
        if "records" in parsed and isinstance(parsed["records"], list):
            return parsed["records"]
        return [parsed]
    if isinstance(parsed, list):
        return parsed
    raise ValueError("Parsed JSON is neither a list nor a dictionary.")


def create_geocoding_columns(record: Dict[str, Any]) -> Dict[str, Any]:
    if record.get("record_type") != "SPREADING":
        record["geo_address"] = "N/A - SOURCE WELL"
        record["map_status"] = "SKIP_WELL"
        return record

    county = str(record.get("app_county", "")).strip()
    township = str(record.get("app_township", "")).strip()
    road = str(record.get("app_road", "")).strip()
    record["geo_address"] = ", ".join(part for part in [road, township, county, "Ohio"] if part)
    record["map_status"] = "READY_FOR_GEOCODING" if road else "MISSING_ROAD"
    return record


DATE_SPLIT_PATTERN = re.compile(r"\s*(?:,|;|\n|\band\b|\&)\s*")


def normalize_date(date_text: str, report_year: Any = "") -> str:
    date_text = str(date_text).strip()
    if not date_text:
        return ""
    has_year = bool(re.search(r"\b\d{4}\b", date_text))
    candidate = date_text if has_year else f"{date_text}/{report_year}".strip("/")
    try:
        parsed_date = date_parser.parse(candidate, fuzzy=True, default=date_parser.parse("01/01/2000"))
        return parsed_date.strftime("%m/%d/%Y")
    except Exception:
        return date_text


def expand_records_by_date(record: Dict[str, Any]) -> List[Dict[str, Any]]:
    if record.get("record_type") != "SPREADING":
        return [record]
    raw_date = str(record.get("app_date", "")).strip()
    if not raw_date:
        return [record]

    parts = [part for part in DATE_SPLIT_PATTERN.split(raw_date) if part]
    if len(parts) <= 1:
        record["app_date"] = normalize_date(raw_date, record.get("report_year", ""))
        return [record]

    expanded = []
    for part in parts:
        new_record = dict(record)
        new_record["app_date"] = normalize_date(part, record.get("report_year", ""))
        expanded.append(new_record)
    return expanded


def flag_data_quality(record: Dict[str, Any]) -> Dict[str, Any]:
    flags = []
    if record.get("record_type") == "SPREADING":
        if not str(record.get("app_road", "")).strip():
            flags.append("missing_road")
        if not str(record.get("app_date", "")).strip():
            flags.append("missing_date")
        volume_text = str(record.get("app_volume_bbl", "")).replace(",", "").strip()
        if volume_text:
            try:
                if float(volume_text) < 0:
                    flags.append("negative_volume")
            except ValueError:
                flags.append("volume_not_numeric")
    record["quality_flags"] = ";".join(flags)
    return record


def extract_records_from_report(file_path: Path | str, content: str, model_name: str, api_key: Optional[str] = None) -> List[Dict[str, Any]]:
    from litellm import completion

    file_path = Path(file_path)
    api_key = api_key or os.getenv(LLM_CONFIG["api_key_env"])
    if not api_key:
        raise ValueError("No API key is available for LiteLLM parsing.")

    schema_hint = json.dumps({"records": [{field: "" for field in TARGET_FIELDS}]}, indent=2)
    user_prompt = f"""
Schema:
{schema_hint}

Source file: {file_path.name}

OCR text:
{content}
""".strip()

    response = completion(
        model=model_name,
        api_key=api_key,
        messages=[
            {"role": "system", "content": PARSING_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
    )

    raw_records = parse_model_json(response.choices[0].message.content)
    cleaned_records = []
    for raw_record in raw_records:
        record = {field: raw_record.get(field, "") for field in TARGET_FIELDS}
        record["source_file"] = file_path.name
        for dated_record in expand_records_by_date(record):
            dated_record = create_geocoding_columns(dated_record)
            dated_record = flag_data_quality(dated_record)
            cleaned_records.append(dated_record)
    return cleaned_records


def process_all_reports(input_dir: Path | str, output_csv: Path | str, model_name: str) -> pd.DataFrame:
    input_dir = Path(input_dir)
    output_csv = Path(output_csv)
    ensure_dir(output_csv.parent)

    txt_files = sorted(input_dir.glob("*.txt"))
    if not txt_files:
        print("No .txt files found.")
        return pd.DataFrame(columns=TARGET_FIELDS + ["geo_address", "map_status", "quality_flags"])

    api_key = load_api_key(LLM_CONFIG["colab_secret_name"], LLM_CONFIG["api_key_env"])
    all_records = []
    for idx, file_path in enumerate(txt_files, start=1):
        print(f"Parsing file {idx}/{len(txt_files)}: {file_path.name}")
        try:
            content = file_path.read_text(encoding="utf-8")
            all_records.extend(extract_records_from_report(file_path=file_path, content=content, model_name=model_name, api_key=api_key))
        except Exception as exc:
            print(f"Failed to parse {file_path.name}: {exc}")

    output_df = pd.DataFrame(all_records)
    output_df.to_csv(output_csv, index=False)
    print(f"Saved parsed dataset to: {output_csv}")
    return output_df


## 7) Combine county-level CSV files

This section merges multiple parsed CSV files into one combined dataset. It is especially useful when extraction and parsing were run separately across counties or batches and need to be bound together before cleaning and validation.


In [ ]:
def combine_csv_files(input_folder: Path | str, output_csv: Path | str, pattern: str = "*.csv") -> pd.DataFrame:
    input_folder = Path(input_folder)
    output_csv = Path(output_csv)
    ensure_dir(output_csv.parent)

    csv_files = sorted([path for path in input_folder.glob(pattern) if path.resolve() != output_csv.resolve()])
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {input_folder}")

    combined_df = pd.concat((pd.read_csv(path) for path in csv_files), ignore_index=True)
    combined_df.to_csv(output_csv, index=False)
    print(f"Combined {len(csv_files)} files into: {output_csv}")
    return combined_df


## 8) Data cleaning and road matching

This section standardizes road-related fields and then matches application roads to a master road list using fuzzy matching. Splitting the logic into small helper functions makes the cleaning rules easier to explain in the protocol manuscript and easier to revise later.


In [ ]:
ROAD_ABBREVIATIONS = {
    r"\bcounty road\b": "co rd",
    r"\broad\b": "rd",
    r"\bstreet\b": "st",
    r"\bavenue\b": "ave",
    r"\bdrive\b": "dr",
    r"\blane\b": "ln",
    r"\bhighway\b": "hwy",
    r"\broute\b": "rte",
    r"\btownship\b": "twp",
}


def normalize_text(value: Any) -> str:
    value = "" if pd.isna(value) else str(value)
    value = value.upper().strip()
    value = re.sub(r"\s+", " ", value)
    return value


def normalize_road_name(value: Any) -> str:
    value = normalize_text(value)
    value = re.sub(r"[^A-Z0-9\s]", " ", value)
    for pattern, replacement in ROAD_ABBREVIATIONS.items():
        value = re.sub(pattern, replacement.upper(), value, flags=re.IGNORECASE)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def build_master_index(master_df: pd.DataFrame, county_col: str, township_col: str, road_col: str) -> Dict[str, Dict[str, List[str]]]:
    master_index = {}
    for county, county_group in master_df.groupby(county_col):
        county_key = normalize_text(county)
        master_index[county_key] = {}
        for township, township_group in county_group.groupby(township_col):
            township_key = normalize_text(township)
            roads = sorted({normalize_road_name(value) for value in township_group[road_col].dropna().astype(str)})
            master_index[county_key][township_key] = roads
    return master_index


def find_best_road_match(row: pd.Series, master_index: Dict[str, Dict[str, List[str]]], township_threshold: int = 85, road_threshold: int = 80) -> Dict[str, Any]:
    from thefuzz import fuzz, process

    input_county = normalize_text(row.get("app_county_clean", row.get("app_county", "")))
    input_township = normalize_text(row.get("app_township", ""))
    input_road = normalize_road_name(row.get("app_road", ""))

    empty_result = {
        "Matched_Township": "",
        "Matched_Road": "",
        "Township_Confidence": 0,
        "Match_Confidence": 0,
        "Match_Note": "no_match",
    }

    if input_county not in master_index:
        empty_result["Match_Note"] = "county_not_found"
        return empty_result

    county_lookup = master_index[input_county]
    township_choices = list(county_lookup.keys())
    township_match = process.extractOne(input_township, township_choices, scorer=fuzz.token_sort_ratio)
    if township_match is None:
        empty_result["Match_Note"] = "township_not_found"
        return empty_result

    matched_township, township_score = township_match[0], township_match[1]
    if township_score < township_threshold:
        empty_result["Matched_Township"] = matched_township
        empty_result["Township_Confidence"] = township_score
        empty_result["Match_Note"] = "township_below_threshold"
        return empty_result

    road_choices = county_lookup.get(matched_township, [])
    road_match = process.extractOne(input_road, road_choices, scorer=fuzz.token_sort_ratio) if road_choices else None
    if road_match is None:
        empty_result["Matched_Township"] = matched_township
        empty_result["Township_Confidence"] = township_score
        empty_result["Match_Note"] = "road_not_found"
        return empty_result

    matched_road, road_score = road_match[0], road_match[1]
    match_note = "matched" if road_score >= road_threshold else "road_below_threshold"
    return {
        "Matched_Township": matched_township,
        "Matched_Road": matched_road,
        "Township_Confidence": township_score,
        "Match_Confidence": road_score,
        "Match_Note": match_note,
    }


def clean_and_match_roads(input_csv: Path | str, master_list_path: Path | str, output_csv: Path | str, county_col: str, township_col: str, road_col: str, township_threshold: int = 85, road_threshold: int = 80) -> pd.DataFrame:
    app_df = pd.read_csv(input_csv)
    master_df = pd.read_csv(master_list_path)

    app_df["app_county_clean"] = app_df["app_county"].apply(normalize_text)
    app_df["app_township_clean"] = app_df["app_township"].apply(normalize_text)
    app_df["app_road_clean"] = app_df["app_road"].apply(normalize_road_name)

    master_index = build_master_index(master_df=master_df, county_col=county_col, township_col=township_col, road_col=road_col)

    results = []
    for _, row in tqdm(app_df.iterrows(), total=len(app_df), desc="Matching roads"):
        results.append(find_best_road_match(row=row, master_index=master_index, township_threshold=township_threshold, road_threshold=road_threshold))

    result_df = pd.DataFrame(results)
    final_df = pd.concat([app_df, result_df], axis=1)
    output_csv = Path(output_csv)
    ensure_dir(output_csv.parent)
    final_df.to_csv(output_csv, index=False)
    print(f"Saved cleaned and matched dataset to: {output_csv}")
    return final_df


## 9) Example run block

This final section shows how to run the workflow using simple step toggles. That approach keeps the notebook readable while still letting you execute the pipeline in a staged way during methods development or manuscript revision.


In [ ]:
def main() -> None:
    mount_google_drive()

    for path in PATHS.values():
        ensure_dir(path)

    if RUN_STEPS["convert_pdfs"]:
        batch_convert_pdfs(PATHS["raw_pdfs"], PATHS["images"], dpi=300, image_format="PNG")

    if RUN_STEPS["enhance_images"]:
        batch_enhance_images(PATHS["images"], PATHS["images_enhanced"])

    if RUN_STEPS["run_nanonets"]:
        batch_nanonets_extraction(
            input_dir=PATHS["images_enhanced"],
            output_dir=PATHS["ocr_text"] / "nanonets_json",
            endpoint="<NANONETS_ENDPOINT>",
            api_key_env="NANONETS_API_KEY",
        )

    if RUN_STEPS["run_qwen"]:
        batch_qwen_extraction(
            input_dir=QWEN_CONFIG["input_folder"],
            output_dir=QWEN_CONFIG["output_folder"],
            model_name=QWEN_CONFIG["model_name"],
            quantize_4bit=QWEN_CONFIG["quantize_4bit"],
            max_new_tokens=QWEN_CONFIG["max_new_tokens"],
        )

    if RUN_STEPS["parse_text"]:
        process_all_reports(
            input_dir=PATHS["ocr_text"] / "<TEXT_SUBFOLDER>",
            output_csv=PATHS["parsed_csv"] / "<COUNTY_OR_BATCH_OUTPUT>.csv",
            model_name=LLM_CONFIG["model"],
        )

    if RUN_STEPS["merge_csvs"]:
        combine_csv_files(PATHS["parsed_csv"], PATHS["merged_csv"] / "AllCounties_Merged_<DATE>.csv")

    if RUN_STEPS["clean_and_match"]:
        clean_and_match_roads(
            input_csv=PATHS["merged_csv"] / "AllCounties_Merged_<DATE>.csv",
            master_list_path=MATCHING_CONFIG["master_list_path"],
            output_csv=PATHS["final_output"] / "Final_AllCounties_Merged_<DATE>.csv",
            county_col=MATCHING_CONFIG["county_col"],
            township_col=MATCHING_CONFIG["township_col"],
            road_col=MATCHING_CONFIG["road_col"],
            township_threshold=MATCHING_CONFIG["township_threshold"],
            road_threshold=MATCHING_CONFIG["road_threshold"],
        )

main()
